# 🧪 معمل التجارب: قاعدة بيانات المتجهات السحابية Pinecone Vector DB (Lecture 36)
أهلاً بك في معملك التطبيقي! 🚀

**فكرة المعمل ببساطة:**
في هذا الدرس، نختم القسم السابع بالتعرف على أشهر وأقوى قاعدة بيانات متجهات سحابية في العالم: **Pinecone**.
تعتبر Pinecone الخيار المفضل لشركات الـ AI الكبرى ورواد الأعمال لأنها:
1. **مصممة خصيصاً للمتجهات (Vector-Native):** لم تكن قاعدة بيانات عادية وتم تعديلها، بل بُنيت من اليوم الأول للتعامل مع متجهات الذكاء الاصطناعي.
2. **معمارية بدون خوادم (Serverless):** لا تدير خوادم ولا توزع شاردات؛ الفهرس يتوسع تلقائياً على خوادم AWS أو GCP.
3. **سرعة بحث خارقة:** استرجاع دلالي في أقل من 50 ميلي ثانية حتى لو كان لديك مليارات المتجهات!

في هذا المعمل ستتعلم عملياً:
- الحصول على مفتاح `PINECONE_API_KEY` وربط التطبيق بالسحابة.
- فحص وإنشاء الفهرس برمجياً (`ServerlessSpec`) وتحديد الأبعاد المتطابقة مع نموذج التضمين.
- تخزين المستندات والبيانات الوصفية (Metadata) واسترجاعها بدقة.
- التصفية السحابية المتقدمة واستخراج درجات الثقة.
- بناء مسار RAG متكامل باستخدام **LCEL** ونموذج **Groq** فائق السرعة.

> 💡 **طريقة التدريب في المعمل:**
> اقرأ الشرح المبسط ⬅️ شوف الكود المرجعي الجاهز ⬅️ واكتب الكود وجربه بإيدك في الخلية الفارغة المخصصة لك تحت كل خطوة!

### 1. استيراد المكتبات والتحقق من مفتاح Pinecone
**الفكرة ببساطة:**
نقوم بتحميل متغيرات البيئة من ملف `.env`، والتحقق من توفر مفتاح `PINECONE_API_KEY` الذي يمكن الحصول عليه مجاناً من منصة [app.pinecone.io](https://app.pinecone.io).
نوفر أيضاً محاكياً تعليمياً ذكياً في حال لم تكن قد سجلت حسابك بعد، حتى تختبر الكود ومسار الـ RAG فوراً.

In [ ]:
# 1. import standard libraries and load environment variables
import os
import time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# check pinecone credential
pinecone_api_key = os.getenv("PINECONE_API_KEY")
has_credentials = bool(pinecone_api_key and len(pinecone_api_key) > 5)

print("✅ Libraries imported successfully!")
if has_credentials:
    print(f"🌲 Pinecone API Key loaded: {pinecone_api_key[:8]}... (Live Cloud Mode)")
else:
    print("ℹ️ PINECONE_API_KEY not found in .env.")
    print("👉 Get your free key at: https://app.pinecone.io")
    print("💡 Running in local demonstration mode.")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب فحص وقراءة مفتاح Pinecone بنفسك هنا):



### 2. تجهيز نموذج التضمين وحساب أبعاد المتجهات
**الفكرة ببساطة:**
في Pinecone، **يجب أن يتطابق بُعد الفهرس (Dimension) تماماً مع حجم مخرجات نموذج التضمين**.
لذلك نستخدم دالة `len(embeddings.embed_query("test"))` لنحدد البعد ديناميكياً (مثلاً 384 لموديل HuggingFace المحلي، أو 1536 لموديل OpenAI).

In [ ]:
# initialize embedding model (local HuggingFace with OpenAI fallback)
if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("Using OpenAI Embeddings (text-embedding-3-small)")
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    print("Using local HuggingFace Embeddings (all-MiniLM-L6-v2)")

# dynamically detect embedding vector dimension
sample_vector = embeddings.embed_query("testing embedding dimension")
embedding_dim = len(sample_vector)

print(f"📏 Detected Embedding Dimension: {embedding_dim}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب قياس أبعاد نموذج التضمين بنفسك هنا):



### 3. إنشاء الفهرس السحابي برمجياً (`ServerlessSpec`)
**الفكرة ببساطة:**
بدل ما تفتح الموقع وتنشئ الفهرس بالماوس، نقدر بالبايثون نتصل بكلاس `Pinecone` ونسأله:
"هل الفهرس `rag-index` موجود؟"
لو مش موجود، ننشئه فوراً بمواصفات السيرفرلس (`cloud="aws"`, `region="us-east-1"`) وبنفس أبعاد التضمين ومقياس التشابه `cosine`!

In [ ]:
index_name = "rag-index"

if has_credentials:
    from pinecone import Pinecone, ServerlessSpec
    
    # initialize pinecone client instance
    pc = Pinecone(api_key=pinecone_api_key)
    
    # check if index already exists
    existing_indexes = [idx.name for idx in pc.list_indexes()]
    print(f"Existing Cloud Indexes: {existing_indexes}")
    
    if index_name not in existing_indexes:
        print(f"Creating new Serverless Index: '{index_name}' (dim={embedding_dim})...")
        pc.create_index(
            name=index_name,
            dimension=embedding_dim,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        # wait for cloud index provisioning
        while not pc.describe_index(index_name).status["ready"]:
            time.sleep(1)
        print("✅ Index created and ready on AWS!")
    else:
        print(f"✅ Index '{index_name}' already exists and is active.")
        
    # connect to pinecone index
    pinecone_index = pc.Index(index_name)
    print(f"📊 Index Stats: {pinecone_index.describe_index_stats()['total_vector_count']} vectors.")
else:
    print(f"💡 Simulation mode: '{index_name}' configured locally.")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب فحص وإنشاء فهرس Pinecone برمجياً بنفسك هنا):



### 4. ربط الفهرس مع لانج تشين (`PineconeVectorStore`)
**الفكرة ببساطة:**
نستخدم كلاس `PineconeVectorStore` من مكتبة `langchain-pinecone`، ونمرر له الفهرس السحابي ونموذج التضمين ليكون جاهزاً لاستقبال المستندات والبحث فيها.

In [ ]:
from langchain_pinecone import PineconeVectorStore

if has_credentials:
    vector_store = PineconeVectorStore(
        index=pinecone_index,
        embedding=embeddings
    )
    print("✅ PineconeVectorStore connected to LIVE Cloud Index!")
else:
    from langchain_core.vectorstores import InMemoryVectorStore
    vector_store = InMemoryVectorStore(embeddings)
    print("✅ Initialized vector store demonstration engine.")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تهيئة كائن PineconeVectorStore بنفسك هنا):



### 5. تجهيز قائمة المستندات المتنوعة مع البيانات الوصفية (Metadata)
**الفكرة ببساطة:**
بنجهز 10 مستندات تغطي مجالات متنوعة (تغريدات تقنية، أخبار الطقس، أرقام البورصة، ومفاهيم الذكاء الاصطناعي)، مع تزويد كل مستند بحقول `metadata` واضحة لتمكين الفلترة لاحقاً.

In [ ]:
from langchain_core.documents import Document

# 10 multi-domain sample documents matching lecture curriculum
sample_documents = [
    Document(
        page_content="I had delicious chocolate chip pancakes and scrambled eggs for breakfast.",
        metadata={"source": "tweet", "topic": "food", "author": "foodie_99"}
    ),
    Document(
        page_content="Building an exciting enterprise generative AI project using LangChain and LangGraph.",
        metadata={"source": "tweet", "topic": "ai", "author": "dev_guru"}
    ),
    Document(
        page_content="The weather forecast for tomorrow indicates cloudy skies with afternoon rain showers.",
        metadata={"source": "news", "topic": "weather", "urgency": "low"}
    ),
    Document(
        page_content="Stock market indices surged today driven by strong quarterly earnings in tech sectors.",
        metadata={"source": "news", "topic": "finance", "urgency": "medium"}
    ),
    Document(
        page_content="A balanced breakfast containing healthy proteins and fruits helps maintain daytime focus.",
        metadata={"source": "health_blog", "topic": "food", "author": "nutritionist"}
    ),
    Document(
        page_content="Stealing from the bank or committing wire fraud is a serious federal crime.",
        metadata={"source": "news", "topic": "legal", "urgency": "high"}
    ),
    Document(
        page_content="Pinecone is a cloud-native vector database designed for high availability and sub-millisecond search at scale.",
        metadata={"source": "tech_blog", "topic": "ai", "author": "cloud_arch"}
    ),
    Document(
        page_content="Severe tropical storm warnings were issued along coastal regions with high wind speeds.",
        metadata={"source": "news", "topic": "weather", "urgency": "high"}
    ),
    Document(
        page_content="LangChain Expression Language (LCEL) simplifies building streaming conversational RAG pipelines.",
        metadata={"source": "tech_blog", "topic": "ai", "author": "dev_guru"}
    ),
    Document(
        page_content="Central banks announced updated interest rate guidance following the economic policy summit.",
        metadata={"source": "news", "topic": "finance", "urgency": "medium"}
    ),
]

print(f"✅ Prepared {len(sample_documents)} sample documents:")
for idx, doc in enumerate(sample_documents[:3], start=1):
    print(f"  [{idx}] ({doc.metadata['source']:<10} | {doc.metadata['topic']:<8}): '{doc.page_content[:55]}...'")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب إنشاء قائمة مستندات مع الـ metadata بنفسك هنا):



### 6. رفع المستندات لسحابة Pinecone (`add_documents`)
**الفكرة ببساطة:**
بنرفع المستندات بـ `vector_store.add_documents`، فتقوم السحابة بحساب الـ Embeddings وحفظها في فهرس Pinecone على AWS فورياً، ونستعرض المعرفات الناتجة.

In [ ]:
# ingest documents into pinecone
doc_ids = vector_store.add_documents(documents=sample_documents)

print(f"✅ Ingested {len(doc_ids)} documents into Pinecone!")
print("🔑 Sample Generated Document IDs:")
for idx, doc_id in enumerate(doc_ids[:3], start=1):
    print(f"  [{idx}] {doc_id}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب رفع المستندات واستعراض المعرفات بنفسك هنا):



### 7. البحث الدلالي السحابي (Similarity Search)
**الفكرة ببساطة:**
بنسأل عن حالة الطقس: *"Will it be hot or rainy tomorrow?"* ونطلب أفضل نتيجتين (`k=2`).
يقوم محرك Pinecone بحساب التشابه وإرجاع أدق النصوص المرتبطة دلالياً بالموضوع.

In [ ]:
# standard similarity search query
weather_query = "Will it be hot or rainy tomorrow?"
weather_results = vector_store.similarity_search(weather_query, k=2)

print(f"❓ Query: '{weather_query}'\n")
print(f"🔍 Top {len(weather_results)} Results:\n")
for idx, doc in enumerate(weather_results, start=1):
    print(f"  [{idx}] Source: {doc.metadata.get('source')} | Topic: {doc.metadata.get('topic')}")
    print(f"      Content: '{doc.page_content}'\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تشغيل بحث دلالي بسؤال من اختيارك بنفسك هنا):



### 8. التصفية بالبيانات الوصفية على السحابة (Metadata Filtering)
**الفكرة ببساطة:**
ميزة Pinecone الجبارة هي الـ Metadata Filtering فائق السرعة؛ نطلب منه البحث داخل التغريدات فقط بـ `filter={"source": "tweet"}`، فيستبعد أي مقال إخباري أو علمي حتى لو كان متطابقاً في الكلمات!

In [ ]:
# similarity search with metadata filter
tweet_query = "What framework is best for generative AI?"

filter_condition = {"source": "tweet"} if has_credentials else (lambda d: d.metadata.get("source") == "tweet")

tweet_results = vector_store.similarity_search(
    query=tweet_query,
    k=2,
    filter=filter_condition
)

print(f"🔍 Query: '{tweet_query}' (Filter: source == 'tweet')\n")
print(f"✅ Filtered Results count: {len(tweet_results)}\n")
for idx, doc in enumerate(tweet_results, start=1):
    print(f"  [{idx}] Source: {doc.metadata.get('source')} | Author: {doc.metadata.get('author')}")
    print(f"      Content: '{doc.page_content}'\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب البحث مع فلترة metadata من اختيارك هنا):



### 9. البحث مع استخراج درجات التطابق (Similarity Search with Score)
**الفكرة ببساطة:**
باستخدام `similarity_search_with_score`، بنحصل على النص المسترجع ومعه نسبة التطابق الدلالي (Score) لتقييم مدى ثقة النتيجة بالأرقام.

In [ ]:
# query returning (document, score) pairs
legal_query = "What are the legal punishments for committing bank fraud?"
scored_results = vector_store.similarity_search_with_score(legal_query, k=2)

print(f"❓ Query: '{legal_query}'\n")
print("📊 Relevance Scored Results:\n")
for idx, (doc, score) in enumerate(scored_results, start=1):
    print(f"  Rank #{idx} | Similarity Score: {score:.4f}")
    print(f"    Topic:   {doc.metadata.get('topic')} | Source: {doc.metadata.get('source')}")
    print(f"    Content: '{doc.page_content}'\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استخراج درجات التطابق وفحص نسبتها بنفسك هنا):



### 10. بناء مسار RAG كامل باستخدام LCEL ونموذج Groq السريع
**الفكرة ببساطة:**
بنربط مسترجع Pinecone السحابي بموديل الذكاء الاصطناعي فائق السرعة **Groq** وقالب الـ Prompt بمشغل الربط `|` في LCEL للحصول على إجابة نهائية دقيقة وموثقة بالسياق.

In [ ]:
# import lcel components and groq chat model
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# helper function to format context chunks
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# context-grounded prompt template
template = """You are an enterprise AI assistant.
Answer the user's question using ONLY the provided context. If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# initialize fast Groq LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)

# convert vector store to retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# assemble full lcel rag chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# execute test query
rag_query = "What is Pinecone and why is it used in enterprise AI?"
answer = rag_chain.invoke(rag_query)

print(f"❓ Question: {rag_query}\n")
print(f"💬 Answer from Pinecone RAG:\n{answer}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب ربط السلسلة وتشغيل استعلام RAG كامل بنفسك هنا):



## 🎯 ملخص المعمل السريع (Lecture 36):
1. **Pinecone Vector DB:** قاعدة بيانات متجهات سحابية متخصصة ومصممة من الصفر لمعالجة متجهات الذكاء الاصطناعي بسرعة خارقة.
2. **`ServerlessSpec`:** تحدد مزود السحابة والمنطقة الجغرافية (مثل `AWS us-east-1`) ليعمل الفهرس تلقائياً بدون إدارة خوادم.
3. **تطابق الأبعاد:** يجب دائماً أن يتطابق بُعد الفهرس (`dimension`) مع أبعاد نموذج التضمين المستخدم.
4. **التصفية السحابية السريعة:** تدعم `filter={"source": "tweet"}` مباشرة على خوادم Pinecone.
5. **الخيار المثالي للـ CV:** دمج Pinecone في مشروعك يثبت لمدراء التوظيف قدرتك على بناء تطبيقات سحابية إنتاجية حقيقية (Production-Ready)!